Mount data from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import and Install libraries

In [ ]:
# Video processing
!pip install opencv-python tqdm

# Audio processing
!pip install librosa

# Data manipulation & EDA
import pandas as pd
import numpy as np

# Video & audio libraries
import cv2
import os
import librosa

# For progress bar
from tqdm import tqdm

Test the path

In [ ]:
video_dir = "/content/drive/MyDrive/Colab Notebooks/Real Life Violence Dataset/NonViolence"
video_list = [f for f in os.listdir(video_dir) if f.endswith((".mp4", ".avi", ".mov"))]
video_list2 = [f for f in os.listdir("/content/drive/MyDrive/Colab Notebooks/Real Life Violence Dataset/Violence") if f.endswith(((".mp4", ".avi", ".mov")))]

print(f"Found {len(video_list)} non violent videos")
print(f"Found {len(video_list2)} violent videos")

Found 1000 non violent videos
Found 1000 violent videos


EDA video, first step: Meta data

In [ ]:
def get_video_metadata(video_path):
    import cv2, os
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    video_id = video_path.split("/")[-1]
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    resolution = f"{width}x{height}"
    duration_sec = frame_count / fps if fps > 0 else 0
    cap.release()

    file_size_MB = os.path.getsize(video_path) / (1024*1024)

    return {
        "video_id": video_id,
        "file_path": video_path,
        "duration_sec": duration_sec,
        "frame_count": frame_count,
        "fps": fps,
        "resolution": resolution,
        "file_size_MB": file_size_MB
    }
# demo example usage

video_path = "/content/drive/MyDrive/Colab Notebooks/Real Life Violence Dataset/NonViolence/NV_8.mp4"

# Lấy metadata
metadata = get_video_metadata(video_path)

# Kiểm tra kết quả
if metadata is not None:
    print("Video metadata:")
    for key, value in metadata.items():
        print(f"{key}: {value}")
else:
    print("Lỗi: không thể mở video.")

Video metadata:
video_id: NV_8.mp4
file_path: /content/drive/MyDrive/Colab Notebooks/Real Life Violence Dataset/NonViolence/NV_8.mp4
duration_sec: 5.12
frame_count: 128
fps: 25.0
resolution: 1280x720
file_size_MB: 1.2407331466674805


Extract video - feature

In [ ]:
import cv2
import numpy as np
from moviepy.editor import VideoFileClip

# ---------- Frame-based feature functions ----------

def compute_mean_brightness(frames):
    gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
    return np.mean([np.mean(f) for f in gray])

def compute_std_brightness(frames):
    gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
    return np.mean([np.std(f) for f in gray])

def compute_motion_intensity(frames):
    motion = []
    for i in range(1, len(frames)):
        prev_gray = cv2.cvtColor(frames[i-1], cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(curr_gray, prev_gray)
        motion.append(np.mean(diff))
    return np.mean(motion) if motion else 0

def compute_scene_change_count(frames, threshold=30):
    scene_changes = 0
    if not frames:
        return 0

    prev_hist = cv2.calcHist([cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)], [0], None, [256], [0,256])
    prev_hist = cv2.normalize(prev_hist, prev_hist).flatten()

    for i in range(1, len(frames)):
        curr_hist = cv2.calcHist([cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)], [0], None, [256], [0,256])
        curr_hist = cv2.normalize(curr_hist, curr_hist).flatten()
        diff = cv2.compareHist(prev_hist, curr_hist, cv2.HISTCMP_BHATTACHARYYA)
        if diff > threshold/100:  # scale threshold to 0-1
            scene_changes += 1
        prev_hist = curr_hist
    return scene_changes

# ---------- Audio-based feature functions ----------

def load_audio_from_video(video_path, sr=22050):
    """Load audio safely; return None nếu không đọc được"""
    try:
        clip = VideoFileClip(video_path)
        if clip.audio is None:
            return None, None
        audio_array = clip.audio.to_soundarray(fps=sr)
        # kiểm tra kiểu trả về
        if not isinstance(audio_array, (list, np.ndarray)):
            return None, None
        if audio_array.ndim > 1:
            audio_array = np.mean(audio_array, axis=1)  # stereo -> mono
        return audio_array, sr
    except Exception as e:
        print(f"Audio load error: {e}")
        return None, None

def compute_audio_energy(audio, sr):
    return np.mean(audio**2)

def compute_audio_std(audio, sr):
    return np.std(audio**2)

# ---------- Wrapper function ----------

def compute_video_features(video_path):
    """
    Load video + audio once, compute all video-level features.
    Return dictionary of features.
    """
    features = {}

    # --- Load video frames ---
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()

    # --- Compute frame-based features ---
    features['mean_brightness'] = compute_mean_brightness(frames) if frames else None
    features['std_brightness'] = compute_std_brightness(frames) if frames else None
    features['motion_intensity'] = compute_motion_intensity(frames) if frames else None
    features['scene_change_count'] = compute_scene_change_count(frames) if frames else None

    # --- Load audio ---
    audio, sr = load_audio_from_video(video_path)
    if audio is not None:
        features['audio_energy'] = compute_audio_energy(audio, sr)
        features['audio_std'] = compute_audio_std(audio, sr)
    else:
        features['audio_energy'] = None
        features['audio_std'] = None

    return features

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [ ]:
video_path = "/content/drive/MyDrive/Colab Notebooks/Real Life Violence Dataset/Violence/V_8.mp4"
video_features = compute_video_features(video_path)
print(video_features)

Audio load error: arrays to stack must be passed as a "sequence" type such as list or tuple.
{'mean_brightness': np.float64(122.71800605369462), 'std_brightness': np.float64(57.51960361174673), 'motion_intensity': np.float64(11.223141459757127), 'scene_change_count': 0, 'audio_energy': None, 'audio_std': None}


Final wrapper

In [ ]:
import cv2
import numpy as np

def compute_video_features_fast(video_path, frame_skip=3, resize_dim=(224, 224)):
    """
    Phiên bản nhanh: chỉ frame-based features, giảm RAM + thời gian.
    frame_skip: lấy 1 frame mỗi 'frame_skip' frame
    resize_dim: resize frame để giảm memory
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video {video_path}")
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    resolution = f"{width}x{height}"
    duration_sec = frame_count / fps if fps > 0 else 0
    file_size_MB = int(cap.get(cv2.CAP_PROP_POS_MSEC)) / (1024*1024) if cap.get(cv2.CAP_PROP_POS_MSEC)>0 else 0
    video_id = video_path.split("/")[-1]

    # Khởi tạo biến để tính incrementally
    brightness_sum = 0
    brightness_sq_sum = 0
    motion_sum = 0
    scene_changes = 0
    prev_hist = None

    count = 0
    prev_gray = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_skip != 0:
            count += 1
            continue

        frame = cv2.resize(frame, resize_dim)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Brightness
        brightness = np.mean(gray)
        brightness_sum += brightness
        brightness_sq_sum += brightness**2

        # Motion
        if prev_gray is not None:
            diff = cv2.absdiff(gray, prev_gray)
            motion_sum += np.mean(diff)
        prev_gray = gray

        # Scene change
        hist = cv2.calcHist([gray], [0], None, [256], [0,256])
        hist = cv2.normalize(hist, hist).flatten()
        if prev_hist is not None:
            diff_hist = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_BHATTACHARYYA)
            if diff_hist > 0.3:  # threshold
                scene_changes += 1
        prev_hist = hist

        count += 1

    cap.release()

    n_frames = count // frame_skip if count>0 else 1
    mean_brightness = brightness_sum / n_frames
    std_brightness = np.sqrt(brightness_sq_sum / n_frames - mean_brightness**2)
    motion_intensity = motion_sum / max(n_frames-1, 1)

    return {
        "video_id": video_id,
        "file_path": video_path,
        "duration_sec": duration_sec,
        "frame_count": frame_count,
        "fps": fps,
        "resolution": resolution,
        "file_size_MB": file_size_MB,
        "mean_brightness": mean_brightness,
        "std_brightness": std_brightness,
        "motion_intensity": motion_intensity,
        "scene_change_count": scene_changes
    }



Dataframe initialization (2nd time)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

dataset_dir = "/content/drive/MyDrive/Colab Notebooks/Real Life Violence Dataset/"
categories = {"Violence": 1, "NonViolence": 0}
batch_size = 100
batch_counter = 0

for category, label in categories.items():
    category_path = os.path.join(dataset_dir, category)
    video_list = [f for f in os.listdir(category_path) if f.endswith((".mp4", ".avi", ".mov"))]

    for i in range(0, len(video_list), batch_size):
        batch_videos = video_list[i:i+batch_size]
        all_features = []

        for video_file in tqdm(batch_videos, desc=f"Processing {category} batch {i//batch_size+1}"):
            video_path = os.path.join(category_path, video_file)
            metadata = get_video_metadata(video_path)
            if metadata is None:
                continue
            features = compute_video_features_fast(video_path)
            features['label'] = label
            combined = {**metadata, **features}
            all_features.append(combined)

        # Save batch CSV
        df_batch = pd.DataFrame(all_features)
        batch_counter += 1
        df_batch.to_csv(f"vdp_dataset_batch{batch_counter}.csv", index=False)
        print(f"Saved batch {batch_counter} with {len(df_batch)} videos")

Processing Violence batch 1:  12%|█▏        | 12/100 [00:27<05:18,  3.62s/it]WARNING:py.warnings:/tmp/ipython-input-2667283109.py:71: RuntimeWarning: invalid value encountered in sqrt
  std_brightness = np.sqrt(brightness_sq_sum / n_frames - mean_brightness**2)

Processing Violence batch 1: 100%|██████████| 100/100 [00:57<00:00,  1.75it/s]


Saved batch 1 with 100 videos


Processing Violence batch 2: 100%|██████████| 100/100 [00:41<00:00,  2.39it/s]


Saved batch 2 with 100 videos


Processing Violence batch 3: 100%|██████████| 100/100 [01:10<00:00,  1.42it/s]


...

Saved batch 18 with 100 videos


Processing NonViolence batch 9: 100%|██████████| 100/100 [00:26<00:00,  3.76it/s]


Saved batch 19 with 100 videos


Processing NonViolence batch 10: 100%|██████████| 100/100 [00:29<00:00,  3.44it/s]

Saved batch 20 with 100 videos


Merge all CSVs

In [ ]:
import pandas as pd
import glob

# Step 1: Get a list of all CSV files matching the pattern
file_list = glob.glob('vdp_dataset_batch*.csv')  # Adjust path if needed

# Step 2: Read and concatenate all CSV files
df_list = [pd.read_csv(file) for file in file_list]
merged_df = pd.concat(df_list, ignore_index=True)

# Step 3: Optional - save the merged file
merged_df.to_csv('vdp_dataset_merged.csv', index=False)

# Step 4: Optional - display some info
print(f"Merged {len(file_list)} files")
print(merged_df.shape)
merged_df.head()

Merged 20 files
(2000, 12)


,video_id,file_path,duration_sec,frame_count,fps,resolution,file_size_MB,mean_brightness,std_brightness,motion_intensity,scene_change_count,label
0,NV_620.mp4,/content/drive/MyDrive/Colab Notebooks/Real Li...,5.0,145,29.0,224x224,0,32.547015,13.589459,9.632433,0,0
1,NV_629.mp4,/content/drive/MyDrive/Colab Notebooks/Real Li...,5.0,145,29.0,224x224,0,68.936334,10.065124,2.676529,1,0
2,NV_632.mp4,/content/drive/MyDrive/Colab Notebooks/Real Li...,5.0,145,29.0,224x224,0,51.987494,16.345374,3.260601,1,0
3,NV_68.mp4,/content/drive/MyDrive/Colab Notebooks/Real Li...,5.0,150,30.0,224x224,0,104.747443,1.770427,9.478546,1,0
4,NV_628.mp4,/content/drive/MyDrive/Colab Notebooks/Real Li...,5.0,145,29.0,224x224,0,20.340344,3.949642,6.099586,1,0


Add the column file size and more detailed data

In [ ]:
import pandas as pd

# Assuming you already loaded your CSV
df = pd.read_csv("/content/vdp_dataset_merged.csv")

# Split the resolution column into two new columns
df[['Width', 'Height']] = df['resolution'].str.split('x', expand=True).astype(int)

import os

def extract_file_size(file_paths):
    sizes_MB = []
    for path in file_paths:
        try:
            size_bytes = os.path.getsize(path)
            size_MB = size_bytes / (1024 * 1024)  # convert bytes to MB
            sizes_MB.append(size_MB)
        except FileNotFoundError:
            sizes_MB.append(None)  # in case the file is missing
    return sizes_MB

# Apply to your dataframe
df['file_size_MB'] = extract_file_size(df['file_path'])
print(df.head())
df.to_csv('vdp_dataset_merged_final.csv', index=False)

     video_id                                          file_path  \
0  NV_620.mp4  /content/drive/MyDrive/Colab Notebooks/Real Li...   
1  NV_629.mp4  /content/drive/MyDrive/Colab Notebooks/Real Li...   
2  NV_632.mp4  /content/drive/MyDrive/Colab Notebooks/Real Li...   
3   NV_68.mp4  /content/drive/MyDrive/Colab Notebooks/Real Li...   
4  NV_628.mp4  /content/drive/MyDrive/Colab Notebooks/Real Li...   

   duration_sec  frame_count   fps resolution  file_size_MB  mean_brightness  \
0           5.0          145  29.0    224x224      0.146583        32.547015   
1           5.0          145  29.0    224x224      0.117890        68.936334   
2           5.0          145  29.0    224x224      0.126650        51.987494   
3           5.0          150  30.0    224x224      0.368282       104.747443   
4           5.0          145  29.0    224x224      0.162156        20.340344   

   std_brightness  motion_intensity  scene_change_count  label  Width  Height  
0       13.589459          9.6